# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [6]:
!uv remove logger-config
!uv add loguru


Resolved 274 packages in 100ms                                       
Uninstalled 2 packages in 0.95ms
 - logger-config==0.3
 - typer-slim==0.24.0
Resolved 276 packages in 116ms                                       
Prepared 1 package in 5.02s                                              
Installed 1 package in 92ms                                 
 + loguru==0.7.3


In [8]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
input_data_dir = "data/research_papers/"
output_data_dir = "data/research_papers_md/"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!CUDA_VISIBLE_DEVICES=1,2,3,4,5,6,7 python scripts/docparser_v2.py --input-dir {input_data_dir} --output-dir {output_data_dir} -c scripts/docling_v2_config.yaml

2026-03-19 20:54:53.368 | INFO     | __main__:export_document_new_docling:340 - Found 68 PDF files to process
2026-03-19 20:54:53.492 | INFO     | __main__:export_document_new_docling:342 - Using devices: ['cuda:0', 'cuda:1', 'cuda:2', 'cuda:3', 'cuda:4', 'cuda:5', 'cuda:6']
2026-03-19 20:54:59.450 | INFO     | __mp_main__:convert_batch_on_device:249 - [cuda:0] Starting conversion for 10 PDFs
[INFO] 2026-03-19 20:55:00,587 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-19 20:55:00,589 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-03-19 20:55:00,601 [RapidOCR] download_file.py:60: File exists and is valid: /workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-19 20:55:00,601 [RapidOCR] main.py:50: Using /workspace/home/lab/rawhad/auto-customize-llm/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
2026-03-19 20:55:00.821 | INFO     | __

In [10]:
# Step 3: Load Processed Document
import glob

# In our example above docling step produces markdown of all the pdf files in the document_collection
with open(glob.glob(f"{output_data_dir}/*.md")[0], "r") as f:
    text = f.read()

In [11]:
# Step 4: Text Chunking and Dataset Creation

from markdown_it import MarkdownIt
from typing import List
import datasets


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """
    Splits Markdown text into chunks at block-level elements
    (headings, paragraphs, lists, tables, code, blockquotes).
    Adds overlap (in words) between all consecutive chunks.

    Args:
        text: The markdown text to be chunked
        max_tokens: Maximum number of words per chunk
        overlap: Number of overlapping words between consecutive chunks

    Returns:
        List of text chunks with specified overlap
    """

    # Initialize markdown parser to understand document structure
    md = MarkdownIt()
    tokens = md.parse(text)

    # Group tokens into block-level segments to preserve markdown structure
    # This ensures we don't split in the middle of headings, lists, etc.
    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    # Split blocks into chunks with overlap to maintain context continuity
    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                # Emit a complete chunk
                chunks.append(" ".join(current_words))
                # Prepare next buffer with overlap from the end of this chunk
                # This ensures context continuity between chunks
                current_words = current_words[-overlap:] if overlap > 0 else []

    # Add any remaining words as the final chunk
    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


chunks = chunk_markdown(text, max_tokens=5000, overlap=1000)


# Prepare seed data for the SDG-Hub knowledge pipeline.
#
# The seed data requires the following fields:
#   - document_outline: A concise title or summary that accurately represents the entire document.
#     For documents covering multiple themes, consider providing multiple outlines (one per section).
#   - icl_document: A representative sample extract from the document. This may include tables, code snippets, definitions, etc.
#   - icl_query_1, icl_query_2, icl_query_3: Three questions based on the icl_document sample.
#   - domain: The domain or subject area of the document.
#
# The code below creates a HuggingFace Dataset from the document chunks,
# then maps the required ICL fields to each entry, and finally saves the result as a JSONL file.

seed_data = datasets.Dataset.from_dict({"document": chunks})

icl = {
    "document_outline": "The document contains excerpts from FINTRAC regulations designed to combat money laundering and terrorist financing in Canada",
    "icl_document": "## Overview\n\nThis guidance came into effect on June 1, 2021.\n\n\nThis guidance explains the methods that can be used by reporting entities\n(REs) to verify the identity of a person or an entity.\n\n\n## 1. Meaning of verifying the identity of a person or an entity\n\nIt means to use the methods described in this guidance to ensure that the\ninformation in an identification document or from other informational\nsources matches the information that the person or entity provided.\n\n\nVerifying identity is a foundational element of Canada's anti-money\nlaundering and anti-terrorist financing regime and a key component of an\nRE's relationship with clients. It helps you to know your clients and to\nunderstand and assess any risk that may be associated to their\ntransactions or activities.\n\n\n## 2. How to verify the identity of a person\n\nYou can use any of the 5 methods described below to identify a person:\n\n- 2.1 Government-issued photo identification method\n\n- 2.2 Credit file method\n\n- 2.3 Dual-process method\n\n- 2.4 Affiliate or member method\n\n- 2.5 Reliance method\n",
    "icl_query_1": "In Canada, what are the methods for verifying someone's identity?",
    "icl_query_2": "In Canada, why is it important to confirm a client's identity?",
    "icl_query_3": "In Canada, can I use Reliance method to verify identity of a person?",
    "domain": "Finance",
}

# Map the ICL fields to each document chunk (if you want to use the same ICL for all, as shown here)
seed_data = seed_data.map(lambda x: icl)

# Save the seed data to a JSONL file for downstream use
seed_data.to_json("seed_data.jsonl", orient="records", lines=True)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

89685

### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook

In [12]:
seed_data

Dataset({
    features: ['document', 'document_outline', 'icl_document', 'icl_query_1', 'icl_query_2', 'icl_query_3', 'domain'],
    num_rows: 3
})

In [14]:
print(seed_data[0]['document'])

ShinkaEvolve: Towards Open-Ended And Sample-Efficient Program Evolution Robert Tjarko Lange, Yuki Imajuku and Edoardo Cetin Sakana AI We introduce ShinkaEvolve 1 : a new open-source framework leveraging large language models (LLMs) to advance scientific discovery with state-of-the-art performance and unprecedented efficiency. Recent advances in scaling inference time compute of LLMs have enabled significant progress in generalized scientific discovery. These approaches rely on evolutionary agentic harnesses that leverage LLMs as mutation operators to generate candidate solutions. However, current code evolution methods suffer from critical limitations: they are sample inefficient, requiring thousands of samples to identify effective solutions, and remain closed-source, hindering broad adoption and extension. ShinkaEvolve addresses these limitations, introducing three key innovations: a parent sampling technique balancing exploration and exploitation, code novelty rejection-sampling for

In [17]:
import textwrap
wprint = lambda x: print(textwrap.fill(x, width=100))
wprint(seed_data[0]['document'])


ShinkaEvolve: Towards Open-Ended And Sample-Efficient Program Evolution Robert Tjarko Lange, Yuki
Imajuku and Edoardo Cetin Sakana AI We introduce ShinkaEvolve 1 : a new open-source framework
leveraging large language models (LLMs) to advance scientific discovery with state-of-the-art
performance and unprecedented efficiency. Recent advances in scaling inference time compute of LLMs
have enabled significant progress in generalized scientific discovery. These approaches rely on
evolutionary agentic harnesses that leverage LLMs as mutation operators to generate candidate
solutions. However, current code evolution methods suffer from critical limitations: they are sample
inefficient, requiring thousands of samples to identify effective solutions, and remain closed-
source, hindering broad adoption and extension. ShinkaEvolve addresses these limitations,
introducing three key innovations: a parent sampling technique balancing exploration and
exploitation, code novelty rejection-sampling fo

In [18]:
wprint(seed_data[1]['document'])

2021; Sap et al., 2019; Talmor et al., 2018; Zellers et al., 2019). We provide our results below as
a function of load imbalance, showing that ShinkaEvolve's new loss achieves a consistent edge across
our training configurations, growing larger with the value of the 𝜆 coefficient. Figure 8 |
ShinkaEvolve for discovering Mixture-of-Experts Load Balancing Loss Functions. Left: Downstream task
performance across seven benchmarks. Middle: Final perplexity across different missroute fractions.
Right: Load imbalance gradient as a function of the token allocation. ShinkaEvolve's Discovered
Solution. The discovered LBL is a new twist on the established global-batch LBL, which was used for
seeding the evolutionary search. ShinkaEvolve complements this popular LBL with a new term,
specifically targeted toward regularizing the MoE layers with individual under-specialized experts.
Concretely, let 𝑓 ℓ,𝑖 and 𝑃ℓ,𝑖 correspond to the selection frequency and the average router
probabilities for each exp